<a href="https://colab.research.google.com/github/kuds/courtside-dynamics/blob/main/notebooks/sb3_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Courtside Dynamics: SB3 training

One notebook for the whole curriculum. Pick an environment (`BallBalance`, `BallBounce`, `WallBall`, `WallBallVolley`, or `WallBallBaseline`) and an algorithm (`SAC` or `PPO`) at the top, then run all cells.

Each environment's defaults (training budget, custom CSV rows, phase labels for the state-machine reward) live in `courtside_dynamics.recipes`, so adding an env is one entry in that registry -- this notebook needs no edits.

## 1. Install

In [ ]:
# Git branch, tag, or commit SHA to install. Keep `main` for normal runs;
# set this to an unmerged PR branch when smoke-testing its environment.
REPO_REF = "main"

!pip install -q "courtside-dynamics[train,notebooks] @ git+https://github.com/kuds/courtside-dynamics@{REPO_REF}"
print(f"Installed courtside-dynamics from ref: {REPO_REF}")


## 2. Choose environment & algorithm

Set `ENV` and `ALGO` here -- everything below picks them up automatically.

* `USE_DRIVE = True` mounts Google Drive so checkpoints, eval npz, replay videos, and learning-curve plots survive a Colab runtime restart. Falls back to local logs if Drive isn't available.
* `QUICK_TEST = True` runs the whole pipeline (training, evaluation, video, plot) in a couple of minutes against a tiny budget. Use it to smoke-test a new runtime before committing to a real run.

### Suggested `n_envs` on Colab L4 (24 GB VRAM) + High-RAM (~51 GB)

These MuJoCo envs are CPU-cheap (tens of microseconds per step) and the MLP policy is small, so the GPU is never the bottleneck. The numbers below assume the default `DummyVecEnv` and the ~8 vCPUs Colab gives you.

| Algo | Suggested `n_envs` | Why |
|------|--------------------|-----|
| SAC  | **8**              | Off-policy. More envs fill the replay buffer faster, and the training helper sets `gradient_steps=-1` so updates scale 1:1 with steps collected regardless of `n_envs`. Past ~8 envs the CPU rollout cost dominates. |
| PPO  | **16**             | On-policy. The rollout buffer is `n_steps * n_envs`, so throughput scales close to linearly with `n_envs`. 16 fits Colab's CPU/RAM budget; bump to 32 if you switch to `SubprocVecEnv`. |

Leaving `N_ENVS = None` uses the recipe's calibrated worker count (every recipe carries one, e.g. 8 for the SAC presets), or -- when you override the recipe's default algorithm -- the per-algo suggestion above. Set an int to override everything.

In [ ]:
ENV = "WallBallBaseline"   # run the next cell to list every recipe; BallBalance | BallBounce | WallBall | WallBallVolley | WallBallBaseline | WallBallBootstrap | HumanoidTennisStage0Intercept | HumanoidTennisStage1AnchoredReturn | HumanoidTennisStage2RandomizedReturn | HumanoidTennisCoopSmoke
ALGO = None           # None uses the recipe default; or set SAC | PPO
USE_DRIVE = True
QUICK_TEST = False

# Master seed forwarded to SB3 and all helper envs. Seeded by default so
# every run is reproducible end-to-end (helper envs get derived,
# non-overlapping seeds); set None for a nondeterministic run. The
# notebook always passes it, so a TOML [train] table may not set seed
# (the loader rejects it loudly).
SEED = 0

# Training budget CEILING. With EARLY_STOP_PATIENCE set below, the run
# stops as soon as eval reward plateaus, so a generous ceiling costs
# nothing -- the first WallBall run peaked at 1.2M steps and spent the
# remaining 10+ GPU-hours collapsing past it. None falls back to the
# recipe default. An explicit value here also overrides the QUICK_TEST
# budget.
TOTAL_TIMESTEPS = None

# Parallel training workers. None uses the CONFIG_FILE / recipe value
# (every recipe now carries its calibrated count, e.g. 8 for the SAC
# presets per the table above); set an int to override.
N_ENVS = None

# Stop training after this many consecutive evaluations without a new
# best mean reward (the same count is used as warm-up, so at least 2x
# this many evaluations happen before a stop can fire). Every recipe
# carries the calibrated default (20; at eval_freq=25k that is 500k env
# steps of no improvement). None defers to the CONFIG_FILE / recipe;
# set an int only to override both.
EARLY_STOP_PATIENCE = None

# Optional per-experiment TOML run config (docs/run_config_file_spec.md).
# Point this at a file in Drive to keep hyperparameters out of notebook
# state: its [train] table sits between the recipe and these explicit
# variables, [train.model_kwargs] deep-merges onto the recipe's bundle,
# and the run records the file's path + sha256 and copies it to
# LOG_DIR/run_config.toml. The next cell lists the packaged starters
# and how to copy one next to your runs.
CONFIG_FILE = None

# Extra kwargs for the SB3 algorithm constructor. None means "add
# nothing": SB3 defaults, improved by the recipe's calibrated bundle
# (e.g. WallBallBootstrap's exploration package) and then by the
# CONFIG_FILE's [train.model_kwargs]. Leave it None: the 20260717 A/B
# showed the old pinned bundle {"ent_coef": 0.02, ...} keeps SAC's
# entropy bonus larger than the task signal and the policy never
# touches the ball (run 025611), while SB3's auto-entropy learned the
# full task (run 165358, 3.23 bounces/ep). A dict set here replaces
# lower layers wholesale -- prefer [train.model_kwargs] in the TOML,
# which merges per key.
MODEL_KWARGS = None

# Post-training endurance audit for the protected WallBall best model.
# These held-out episodes use a separate 5,000-step environment and
# write JSON + per-episode CSV metrics directly into LOG_DIR on Drive.
RUN_LONG_HORIZON_EVAL = True
LONG_HORIZON_EPISODE_LEN = 5_000
LONG_HORIZON_N_EPISODES = 50  # choose 50-100 for a full run
LONG_HORIZON_SEED_START = 10_000


### 2b. Available environments & starter run-configs

Every registered recipe ships with a packaged starter TOML pinned to its
calibrated values. The cell below lists both. To use one: copy it next to
your runs (requires Drive mounted -- run section 3 first), edit the copy in
the Drive UI, and set `CONFIG_FILE` above to its path. `copy_starter_config`
refuses to overwrite an edited copy unless you pass `overwrite=True`.


In [ ]:
from courtside_dynamics.recipes import RECIPES
from courtside_dynamics.run_config import available_run_configs

starters = available_run_configs()
print(f"{'ENV':<38} {'algo':<4} {'steps':>10}  starter config")
for name, recipe in RECIPES.items():
    starter = starters[name].name if name in starters else "-"
    print(f"{name:<38} {recipe.default_algo:<4} {recipe.default_total_timesteps:>10,}  {starter}")
    print(f"{'':<38} {recipe.description}")

# To start from a packaged template (after mounting Drive in section 3):
# from courtside_dynamics.run_config import copy_starter_config
# CONFIG_FILE = copy_starter_config(
#     ENV,
#     "/content/drive/MyDrive/Finding Theta/courtside-dynamics/configs",
# )
# print("Edit in the Drive UI, then re-run from section 5:", CONFIG_FILE)


## 3. Mount Google Drive (optional) and pick a run directory

Each call to `resolve_run_dir(ENV, ALGO)` creates a fresh, timestamped directory so re-runs don't clobber prior artifacts. Layout:

```
<root>/<env>/<algo>/<YYYYMMDD_HHMMSS>/
  best_model.zip          final_model.zip
  evaluations.npz         monitor/*.monitor.csv
  tensorboard/            videos/
  checkpoints/            eval_info.csv
  vec_normalize.pkl       best_vec_normalize.pkl
  config.json             stage_summary.txt
  learning_curve.png      eval_info.png
  training_health.png     best_model.mp4
  best_model_long_horizon_eval.json       # WallBall post-training
  best_model_long_horizon_episodes.csv    # WallBall post-training
```

`checkpoints/` holds periodic full-state snapshots from `CheckpointCallback`; `eval_info.csv` is the long-format mirror of `InfoDictEvalCallback`'s TensorBoard scalars (timestep, metric, value). Two `VecNormalize` snapshots are written: `best_vec_normalize.pkl` captures the obs-normalization stats at the moment `best_model.zip` was saved, while `vec_normalize.pkl` holds the end-of-training stats. `record_best_model_video` prefers the former, so replay normalizes observations exactly the way the best model saw them.

Root is `MyDrive/Finding Theta/courtside-dynamics/training_runs/` when Drive is mounted, otherwise `./logs/`.


In [ ]:
from courtside_dynamics.notebook_utils import mount_drive, resolve_run_dir
from courtside_dynamics.recipes import RECIPES

if USE_DRIVE:
    mount_drive()

ALGO = ALGO or RECIPES[ENV].default_algo
LOG_DIR = resolve_run_dir(ENV, ALGO, use_drive=USE_DRIVE)
print("Logging to:", LOG_DIR)

## 4. Configure Colab GPU

Sets up EGL so MuJoCo can render off-screen on the Colab GPU. No-op outside of Colab.

In [ ]:
from courtside_dynamics.colab_setup import setup_colab
setup_colab()

## 5. Build the training config

`build_train_config` looks up the recipe for `ENV`, fills in the per-env extras (e.g. custom CSV rows for Ball Bounce), and returns a `TrainConfig` ready for `train()`.

In [ ]:
from courtside_dynamics.recipes import RECIPES, build_train_config

print(f"Recipe: {ENV} -> {RECIPES[ENV].description}")

# Explicit keyword arguments beat every other layer (recipe < TOML file
# < quick_test < explicit), so notebook variables are only passed when
# actually set. Every recipe now carries its calibrated worker count, so
# n_envs comes from the recipe (or the TOML) unless N_ENVS overrides it.
overrides = {}
if N_ENVS is not None:
    overrides["n_envs"] = N_ENVS
elif ALGO.upper() != RECIPES[ENV].default_algo.upper():
    # Each recipe's worker count is calibrated for its default
    # algorithm; when overriding the algo, use the section-2 table.
    overrides["n_envs"] = 8 if ALGO.upper() == "SAC" else 16
if EARLY_STOP_PATIENCE is not None:
    overrides["early_stop_patience"] = EARLY_STOP_PATIENCE
if MODEL_KWARGS is not None:
    overrides["model_kwargs"] = MODEL_KWARGS

cfg = build_train_config(
    ENV,
    algo=ALGO,
    log_dir=LOG_DIR,
    total_timesteps=TOTAL_TIMESTEPS,
    quick_test=QUICK_TEST,
    seed=SEED,
    config_file=CONFIG_FILE,
    **overrides,
)

if cfg.run_config_file is not None:
    print(
        f"run config: {cfg.run_config_file.path} "
        f"(sha256 {cfg.run_config_file.sha256[:12]})"
    )

probe_env = cfg.env_fn()
try:
    print(f"spaces: action={probe_env.action_space.shape} observation={probe_env.observation_space.shape}")
finally:
    probe_env.close()

print(
    f"algo={cfg.algo}  total_timesteps={cfg.total_timesteps:,}  "
    f"n_envs={cfg.n_envs}  seed={cfg.seed}  eval_freq={cfg.eval_freq:,}  "
    f"early_stop_patience={cfg.early_stop_patience}  "
    f"checkpoint_freq={cfg.checkpoint_freq:,}  "
    f"video_freq={cfg.video_freq:,}  model_kwargs={cfg.model_kwargs}"
)


## 5b. Live TensorBoard (optional)

Starts an inline TensorBoard tailing `LOG_DIR/tensorboard` so a multi-hour run can be checked mid-flight: eval reward under `eval/`, optimizer health under `train/`, per-eval info metrics under `eval_info/`. Scalars appear after SB3's first metric dump; use the refresh button. Safe to skip -- every scalar shown here is also mirrored to `progress.csv` / `eval_info.csv` and plotted statically in sections 7-8b.

In [ ]:
import os

from tensorboard import notebook as tb_notebook

# notebook.start shlex-parses its args, so the quotes keep Drive
# paths with spaces (MyDrive/Finding Theta/...) intact.
tb_notebook.start(f'--logdir "{os.path.join(LOG_DIR, "tensorboard")}"')


## 6. Train

`train(cfg)` builds vectorized train + eval envs, attaches `EvalCallback`, `VideoRecordCallback`, and `InfoDictEvalCallback`, and runs SB3's `model.learn`. The best policy seen during evaluation is saved to `LOG_DIR/best_model.zip` (with its matching `best_vec_normalize.pkl`), so a later collapse never loses it. With `EARLY_STOP_PATIENCE` set, training stops automatically once evaluations plateau -- the stop reason prints below and `stage_summary.txt` records how many steps of the budget were actually used.

In [ ]:
from courtside_dynamics.training import train

model = train(cfg)

## 6b. Run report

`train()` writes `stage_summary.txt` at the end of every run -- final/best eval, wall-clock duration, throughput, device, and the final `train/*` health metrics -- including interrupted runs (`status: interrupted`). Printing it here attaches the numbers to this notebook session, and it's the first thing to paste when asking "why did this run underperform?".

In [ ]:
from courtside_dynamics.notebook_utils import print_stage_summary

print_stage_summary(LOG_DIR)


## 7. Learning curves

Per-episode training rewards (left) come from `LOG_DIR/monitor/*.monitor.csv`. Deterministic eval rewards (right, mean +/- std) come from `LOG_DIR/evaluations.npz`.

In [ ]:
import os
from courtside_dynamics.notebook_utils import plot_learning_curve

plot_learning_curve(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "learning_curve.png"),
)

## 8. Eval-info curves

One panel per scalar `info` key tracked by `InfoDictEvalCallback` (rally count, paddle / wall touches, phase fractions, ...). `_mean`, `_final`, and `_max` variants are overlaid as separate lines per panel. The data comes from `LOG_DIR/eval_info.csv` (long-format mirror of the TensorBoard scalars, written every eval).

In [ ]:
from courtside_dynamics.notebook_utils import plot_eval_info

plot_eval_info(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "eval_info.png"),
)

## 8b. Training-health curves

SB3's own optimizer diagnostics from `LOG_DIR/tensorboard/progress.csv`. For **SAC**: `ent_coef` (the entropy temperature — watch for it collapsing too fast or sticking high), plus `actor_loss` / `critic_loss` (a diverging critic is the classic failure). For **PPO**: `explained_variance` (below 0 means the value function is worse than predicting the mean), `approx_kl`, `clip_fraction`. These explain a stalled run that the reward curve alone won't.

In [ ]:
from courtside_dynamics.notebook_utils import plot_training_health

plot_training_health(
    LOG_DIR,
    save_path=os.path.join(LOG_DIR, "training_health.png"),
)

## 9. Long-horizon evaluation of the protected WallBall best model

This is separate from training and checkpoint selection. For every recipe backed by `WallBallEnv` (open, volley, or baseline), it reloads `best_model.zip` with its exact `best_vec_normalize.pkl`, validates their metadata and normalization contract, evaluates deterministic held-out seeds in a new 5,000-step environment with the same rally-style preset and canonical normal-only resets, and writes an aggregate JSON plus one-row-per-episode CSV directly into `LOG_DIR` (the mounted Google Drive run folder when `USE_DRIVE=True`). The evaluator records completed-return distributions, explicit survival rates at 2, 3, and 5 returns, the full observed return curve, step survival, termination causes, floor contacts, post-bounce paddle recoveries and completed returns, and reward components. A quick test uses five 1,000-step episodes.

In [ ]:
import json

from courtside_dynamics.envs import WallBallEnv
from courtside_dynamics.notebook_utils import (
    WALL_BALL_LONG_HORIZON_ARTIFACTS,
    evaluate_best_wall_ball,
)
from courtside_dynamics.recipes import RECIPES, make_eval_env_fn

long_horizon_artifacts = ()
is_wall_ball_recipe = issubclass(RECIPES[ENV].env_cls, WallBallEnv)
if is_wall_ball_recipe and RUN_LONG_HORIZON_EVAL:
    long_horizon_artifacts = WALL_BALL_LONG_HORIZON_ARTIFACTS
    if not QUICK_TEST and not 50 <= LONG_HORIZON_N_EPISODES <= 100:
        raise ValueError("LONG_HORIZON_N_EPISODES must be between 50 and 100")

    long_episode_len = 1_000 if QUICK_TEST else LONG_HORIZON_EPISODE_LEN
    long_episode_count = 5 if QUICK_TEST else LONG_HORIZON_N_EPISODES
    long_eval_env_fn = make_eval_env_fn(
        ENV,
        env_overrides={"episode_len": long_episode_len},
    )
    long_eval_seeds = tuple(
        range(
            LONG_HORIZON_SEED_START,
            LONG_HORIZON_SEED_START + long_episode_count,
        )
    )
    long_eval = evaluate_best_wall_ball(
        LOG_DIR,
        long_eval_env_fn,
        algo=cfg.algo,
        episode_len=long_episode_len,
        seeds=long_eval_seeds,
    )
    print(json.dumps({
        "selected_timestep": long_eval["policy"]["best_model_meta"]["timestep"],
        "pair_verification": long_eval["policy"]["pair_verification"],
        "completed_returns": long_eval["metrics"]["completed_returns"],
        "return_survival_curve": long_eval["return_survival_curve"],
        "episode_length": long_eval["metrics"]["episode_length"],
        "floor_bounce_diagnostics": long_eval["floor_bounce_diagnostics"],
        "terminations": long_eval["terminations"],
    }, indent=2))
else:
    print("Long-horizon post-training evaluation is enabled for WallBallEnv recipes only.")


## 9b. Replay the best model

Loads `best_model.zip` from `LOG_DIR`, rolls it out deterministically, encodes the frames as MP4, and embeds the clip in this notebook. The metrics above are saved first, so an optional video-codec failure cannot prevent the endurance audit.

In [ ]:
from courtside_dynamics.notebook_utils import (
    record_best_model_video,
    display_video,
)

video_path = record_best_model_video(
    LOG_DIR,
    cfg.eval_env_fn or cfg.env_fn,
    algo=ALGO,
    video_length=750,
)
display_video(video_path)

## 9c. Artifact audit

Checks `LOG_DIR` against every artifact this notebook should have produced and prints the most likely cause for anything missing (video skipped because moviepy failed, no `best_model.zip` because eval never fired, ...). Run it **before** disconnecting: it's the last chance to re-run a failed cell while the runtime -- and everything not synced to Drive -- still exists.

In [ ]:
from courtside_dynamics.notebook_utils import check_run_artifacts

missing = check_run_artifacts(
    LOG_DIR,
    extra_artifacts=long_horizon_artifacts,
)


## 10. Disconnect Colab runtime

Frees the GPU once the whole pipeline -- training, plots, replay video, long-horizon evaluation, and artifact audit -- has finished, so an unattended Run-All doesn't hold the runtime for hours after the work is done. Everything above is already saved under `LOG_DIR` (on Drive when `USE_DRIVE=True`) and audited in section 9c. Comment the cell out if you want to keep the session alive for interactive inspection. No-op outside Colab.

In [ ]:
# Frees the GPU when the run is over. All artifacts are already on
# disk (and Drive when USE_DRIVE=True) and audited above; comment
# this out to keep the runtime for interactive inspection. No-op
# outside Colab. The delay lets the final cell outputs render.
from courtside_dynamics.notebook_utils import disconnect_runtime

disconnect_runtime(delay_seconds=30)
